In [14]:
import pandas as pd

import json
import numpy as np

In [15]:
with open("arxivData.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Если data — это список словарей
df = pd.DataFrame(data)

# Если нужные поля вложены
df = pd.json_normalize(data, sep='_')  # "Разворачивает" вложенные объекты


In [16]:
import ast
df['terms'] = df['tag'].apply(lambda x: [tag['term'] for tag in ast.literal_eval(x)])


In [17]:
df = df[['title', 'summary', 'terms']]

In [18]:
df

,title,summary,terms
0,Dual Recurrent Attention Units for Visual Ques...,We propose an architecture for VQA which utili...,"[cs.AI, cs.CL, cs.CV, cs.NE, stat.ML]"
1,Sequential Short-Text Classification with Recu...,Recent approaches based on artificial neural n...,"[cs.CL, cs.AI, cs.LG, cs.NE, stat.ML]"
2,Multiresolution Recurrent Neural Networks: An ...,We introduce the multiresolution recurrent neu...,"[cs.CL, cs.AI, cs.LG, cs.NE, stat.ML, I.5.1; I..."
3,Learning what to share between loosely related...,Multi-task learning is motivated by the observ...,"[stat.ML, cs.AI, cs.CL, cs.LG, cs.NE]"
4,A Deep Reinforcement Learning Chatbot,We present MILABOT: a deep reinforcement learn...,"[cs.CL, cs.AI, cs.LG, cs.NE, stat.ML, I.5.1; I..."
...,...,...,...
40995,Nearly Tight Bounds on $\ell_1$ Approximation ...,We study the complexity of learning and approx...,"[cs.LG, cs.DS]"
40996,Concurrent bandits and cognitive radio networks,We consider the problem of multiple users targ...,"[cs.LG, cs.MA]"
40997,A Comparison of Clustering and Missing Data Me...,"In this paper, we compare and analyze clusteri...","[math.NA, cs.LG, 62H30, 91C20, 94A08]"
40998,Applying machine learning to the problem of ch...,Cylindrical algebraic decomposition(CAD) is a ...,"[cs.SC, cs.LG, 68W30, 68T05, O3C10, I.2.6]"


In [19]:
import re

df["terms"] = df["terms"].apply(lambda terms: 
    list(set(subterm.strip() for term in terms for subterm in re.split(r",\s*|;\s*", term)))
)


In [20]:
from collections import Counter

# Собираем все метки в один список
all_labels_flat = [label for terms in df["terms"] for label in terms]

# Считаем частоту
label_counts = Counter(all_labels_flat)



In [21]:
import pandas as pd

label_freq_df = pd.DataFrame(label_counts.items(), columns=["label", "count"])
label_freq_df = label_freq_df.sort_values(by="count", ascending=False)

print(label_freq_df.head(500))  # Топ-500 меток


                     label  count
2                    cs.CV  13902
5                    cs.LG  13735
4                    cs.AI  10481
3                  stat.ML  10326
1                    cs.CL   6417
..                     ...    ...
200                  65C10      2
59         68T30 (Primary)      2
656                  K.4.0      2
671                  49L20      2
197  68Q32 (Primary) 68T05      2

[500 rows x 2 columns]


In [22]:
all_labels = list(set(label for terms in df["terms"] for label in terms))
len(all_labels)

1301

In [23]:
#оставляем только те метки которые встречаются хотя бы 5 раз
filtered_labels = set(label_freq_df[label_freq_df["count"] >= 5]["label"])
df["filtered_terms"] = df["terms"].apply(lambda labels: [label for label in labels if label in filtered_labels])
df = df[df["filtered_terms"].apply(len) > 0]  # Убираем пустые метки

# Обновляем колонку с метками
df = df.drop(columns=["terms"]).rename(columns={"filtered_terms": "terms"})

In [24]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
mlb = MultiLabelBinarizer()
labels = mlb.fit_transform(df["terms"])
num_labels = len(mlb.classes_)

In [32]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import Trainer, TrainingArguments
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import BertTokenizer, BertForSequenceClassification

# Загружаем токенизатор
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Загружаем модель
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=num_labels, problem_type="multi_label_classification"
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [37]:
from skmultilearn.model_selection import iterative_train_test_split
import numpy as np

# Преобразуем тексты и метки в numpy-массивы
texts_np = np.array(df[["title", "summary"]].values.tolist())
labels_np = np.array(labels)

# Стратифицированное разбиение
train_texts, train_labels, val_texts, val_labels = iterative_train_test_split(
    texts_np, labels_np, test_size=0.2
)


In [19]:
# Загружаем токенизатор
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

In [41]:
class MultiLabelDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=320):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        title, summary = self.texts[idx]
        encoding = self.tokenizer(
            title, summary,
            padding = True, truncation=True, max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [42]:
def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    labels = [item["labels"] for item in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = torch.stack(labels)

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [43]:
train_dataset = MultiLabelDataset(train_texts, train_labels, tokenizer)
val_dataset = MultiLabelDataset(val_texts, val_labels, tokenizer)

In [86]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=num_labels, problem_type="multi_label_classification"
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [87]:
for param in model.distilbert.parameters():
    param.requires_grad = False

for layer in model.distilbert.transformer.layer[-2:]:  # Размораживаем последние 2 слоя
    for param in layer.parameters():
        param.requires_grad = True

In [47]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=8,
    weight_decay=0.01,
    warmup_steps = 200,
    lr_scheduler_type = 'reduce_lr_on_plateau',
    optim = 'adamw_torch',
    logging_dir="./logs",
)

/home/jupyter-nenakhov.i/.local/lib/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [24]:
import transformers
transformers.utils.logging.set_verbosity_error()


In [90]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn
)

# Запускаем обучение
trainer.train()

# Сохраняем модель
model.save_pretrained("./multi_label_bert")
tokenizer.save_pretrained("./multi_label_bert")

Epoch,Training Loss,Validation Loss
1,0.068300,0.016264
2,0.014900,0.013624
3,0.013100,0.012636
4,0.012300,0.012227
5,0.011500,0.011806
6,0.011000,0.011660
7,0.010300,0.011655
8,0.009900,0.011603


('./multi_label_bert/tokenizer_config.json',
 './multi_label_bert/special_tokens_map.json',
 './multi_label_bert/vocab.txt',
 './multi_label_bert/added_tokens.json')

In [33]:
def predict(texts, batch_size=16):
    model.eval()
    all_probs = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encodings = tokenizer(
            [text[0] for text in batch], [text[1] for text in batch],
            padding=True, truncation=True, max_length=320, return_tensors="pt"
        )
        with torch.no_grad():
            outputs = model(**encodings)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()
        all_probs.append(probs)

    all_probs = np.vstack(all_probs)
    return (all_probs > 0.5).astype(int)  # Применяем порог 0.5 для бинарных меток


In [22]:
checkpoint = "multi_label_bert"  # Путь к директории с чекпоинтами
model = DistilBertForSequenceClassification.from_pretrained(checkpoint)
tokenizer = DistilBertTokenizer.from_pretrained(checkpoint)

In [27]:
from sklearn.metrics import precision_recall_fscore_support
def evaluate_model(dataset, labels, batch_size=16):
    predictions = predict(dataset, batch_size=batch_size)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="micro")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")


evaluate_model(train_texts, train_labels)
evaluate_model(val_texts, val_labels)

Precision: 0.8816, Recall: 0.6246, F1-score: 0.7311
Precision: 0.7984, Recall: 0.5543, F1-score: 0.6543


In [44]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Загружаем модель
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=num_labels, problem_type="multi_label_classification"
)

In [49]:
# Замораживаем все параметры
for param in model.bert.parameters():
    param.requires_grad = False

# Размораживаем эмбеддинги
for param in model.bert.embeddings.parameters():
    param.requires_grad = True

# Размораживаем последние 2 слоя энкодера
for layer in model.bert.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True


In [51]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn
)

# Запускаем обучение
trainer.train()

# Сохраняем модель
model.save_pretrained("./multi_label_bert_full")
tokenizer.save_pretrained("./multi_label_bert_full")

{'loss': 0.066, 'grad_norm': 0.01235321257263422, 'learning_rate': 5e-05, 'epoch': 0.9823182711198428}
{'eval_loss': 0.02211889624595642, 'eval_runtime': 1091.7721, 'eval_samples_per_second': 7.743, 'eval_steps_per_second': 0.122, 'epoch': 1.0}
{'loss': 0.0204, 'grad_norm': 0.014184418134391308, 'learning_rate': 5e-05, 'epoch': 1.9646365422396856}
{'eval_loss': 0.016917308792471886, 'eval_runtime': 1018.7314, 'eval_samples_per_second': 8.299, 'eval_steps_per_second': 0.131, 'epoch': 2.0}
{'loss': 0.0167, 'grad_norm': 0.009422474540770054, 'learning_rate': 5e-05, 'epoch': 2.9469548133595285}
{'eval_loss': 0.014993718825280666, 'eval_runtime': 883.046, 'eval_samples_per_second': 9.574, 'eval_steps_per_second': 0.151, 'epoch': 3.0}
{'loss': 0.0152, 'grad_norm': 0.010553620755672455, 'learning_rate': 5e-05, 'epoch': 3.9292730844793713}
{'eval_loss': 0.014208106324076653, 'eval_runtime': 890.1151, 'eval_samples_per_second': 9.498, 'eval_steps_per_second': 0.149, 'epoch': 4.0}
{'eval_loss': 

('./multi_label_bert_full/tokenizer_config.json',
 './multi_label_bert_full/special_tokens_map.json',
 './multi_label_bert_full/vocab.txt',
 './multi_label_bert_full/added_tokens.json')

In [52]:
from sklearn.metrics import precision_recall_fscore_support
def evaluate_model(dataset, labels, batch_size=16):
    predictions = predict(dataset, batch_size=batch_size)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="micro")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")


evaluate_model(train_texts, train_labels)
evaluate_model(val_texts, val_labels)

Precision: 0.8419, Recall: 0.5329, F1-score: 0.6527
Precision: 0.7873, Recall: 0.5128, F1-score: 0.6210


In [131]:
from tqdm import tqdm
unk_count = 0
total_tokens = 0

for _, row in tqdm(df.iterrows(), total=len(df)):
    text = str(row["title"]) + " " + str(row["summary"])
    tokens = tokenizer.tokenize(text)
    unk_count += tokens.count("[UNK]")
    total_tokens += len(tokens)

print(f"Всего токенов: {total_tokens}")
print(f"[UNK] токенов: {unk_count}")
print(f"Доля [UNK]: {unk_count / total_tokens:.4f}")

100%|██████████| 41000/41000 [01:56<00:00, 350.95it/s]

Всего токенов: 8608713
[UNK] токенов: 0
Доля [UNK]: 0.0000
